# 最终结构提取

**用途**：在频率流程结束后，提取最终结构（CONTCAR/POSCAR → .vasp）。

- 所有完成的结构 → 输出到 `final_output_dir`
- 被 507 修正的结构 → 额外输出到 `corrected_output_dir`（config 中指定，稍后统一配置）

In [ ]:
import os
import shutil
from pathlib import Path
from ase.io import read, write

In [ ]:
# 从 config 读取
import sys
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
cfg = load_config()
zpe_root = cfg.ZPE_ROOT
corrected_output_dir = getattr(cfg, 'CORRECTED_OUTPUT_DIR', None)

# 主输出目录（与 ZPE 平级的 Frequency_final，可改）
final_output_dir = Path(zpe_root).parent / "Frequency_final"

In [ ]:
def is_corrected(job_dir):
    """是否被 507 修正过（目录内有 _507_corrected 标记）"""
    return (Path(job_dir) / "_507_corrected").exists()


def get_struct_path(job_dir):
    """优先 CONTCAR，否则 POSCAR"""
    p = Path(job_dir)
    if (p / "CONTCAR").exists():
        return p / "CONTCAR"
    if (p / "POSCAR").exists():
        return p / "POSCAR"
    return None


def job_name_from_path(job_dir):
    """从路径生成 job 名"""
    return Path(job_dir).name.replace(" ", "_")

In [ ]:
# 扫描含 OUTCAR 的任务目录（已完成）
outcar_paths = list(Path(zpe_root).rglob("OUTCAR"))
job_dirs = sorted(set(str(p.parent) for p in outcar_paths))

print(f"📌 ZPE: {zpe_root}")
print(f"主输出: {final_output_dir}")
print(f"修正结构输出: {corrected_output_dir or '(未配置，跳过)'}")
print(f"找到 {len(job_dirs)} 个已完成任务\n")
print("=" * 60)

In [ ]:
# 提取到主输出
final_output_dir.mkdir(parents=True, exist_ok=True)
for jd in job_dirs:
    src = get_struct_path(jd)
    if not src:
        continue
    rel = os.path.relpath(jd, zpe_root)
    jn = job_name_from_path(jd)
    target_dir = final_output_dir / Path(rel).parent
    target_dir.mkdir(parents=True, exist_ok=True)
    target_vasp = target_dir / f"{jn}.vasp"
    atoms = read(src, format="vasp")
    write(target_vasp, atoms, vasp5=True)
    print(f"✓ {rel} -> {target_vasp.relative_to(final_output_dir)}")
print(f"\n✔ 主输出已保存到 {final_output_dir}")

In [ ]:
# 被 507 修正的结构 → 单独输出（若已配置 CORRECTED_OUTPUT_DIR）
if corrected_output_dir:
    corrected_path = Path(corrected_output_dir)
    corrected_path.mkdir(parents=True, exist_ok=True)
    n = 0
    for jd in job_dirs:
        if not is_corrected(jd):
            continue
        src = get_struct_path(jd)
        if not src:
            continue
        jn = job_name_from_path(jd)
        target_vasp = corrected_path / f"{jn}.vasp"
        atoms = read(src, format="vasp")
        write(target_vasp, atoms, vasp5=True)
        print(f"✓ [修正] {jn} -> {target_vasp}")
        n += 1
    print(f"\n✔ 修正结构 {n} 个已保存到 {corrected_path}")
else:
    corrected_count = sum(1 for jd in job_dirs if is_corrected(jd))
    if corrected_count > 0:
        print(f"\n💡 有 {corrected_count} 个被修正结构，请在 config.py 中设置 CORRECTED_OUTPUT_DIR 后重新运行")